In [1]:
import pandas  as pd 

In [2]:
from sklearn.model_selection import train_test_split ,GridSearchCV
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler 
from sklearn.pipeline import Pipeline 
from scikeras.wrappers import KerasClassifier  
import tensorflow as tf   
from tensorflow .keras.models import Sequential  
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import EarlyStopping 

ModuleNotFoundError: No module named 'scikeras'

In [ ]:
df=pd.read_csv('Churn_Modelling.csv')
df.head()
df=df.drop(['RowNumber',"CustomerId","Surname"],axis=1) 
df.head()  
la_encoder_gender=LabelEncoder()
df['Gender']=la_encoder_gender.fit_transform(df['Gender'] ) 
df.head() 
from sklearn.preprocessing import OneHotEncoder    
one_geo=OneHotEncoder() 
geo_encoded=one_geo.fit_transform(df[['Geography']]) 
geo_encoded 
geo=pd.DataFrame(geo_encoded.toarray(),columns=one_geo.get_feature_names_out(['Geography']))
df.drop("Geography",axis=1,inplace=True)  
dfn=pd.concat([df,geo],axis=1)  
dfn.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [ ]:
#  saving encoder  
import pickle  as p   
with open('label_encoder_gender.pkl','wb')as file: 
    p.dump(la_encoder_gender,file)  


with open('onehot_encoder_geo.pkl','wb') as file : 
    p.dump(one_geo,file)

In [ ]:

dfn.info()
x=dfn.drop('Exited',axis=1) 
y=dfn['Exited']

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CreditScore        10000 non-null  int64  
 1   Gender             10000 non-null  int32  
 2   Age                10000 non-null  int64  
 3   Tenure             10000 non-null  int64  
 4   Balance            10000 non-null  float64
 5   NumOfProducts      10000 non-null  int64  
 6   HasCrCard          10000 non-null  int64  
 7   IsActiveMember     10000 non-null  int64  
 8   EstimatedSalary    10000 non-null  float64
 9   Exited             10000 non-null  int64  
 10  Geography_France   10000 non-null  float64
 11  Geography_Germany  10000 non-null  float64
 12  Geography_Spain    10000 non-null  float64
dtypes: float64(5), int32(1), int64(7)
memory usage: 976.7 KB


In [ ]:

from sklearn.model_selection import train_test_split 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.20,random_state=42)  



# scaling data 
from sklearn.preprocessing import StandardScaler 
sc=StandardScaler() 
x_train=sc.fit_transform(x_train) 
x_test=sc.transform(x_test)

In [ ]:
with open('scaler.pkl','wb') as file : 
    p.dump(sc,file)

In [ ]:
def create_model(neurons=32, layers=1): 
    model=Sequential() 
    model.add(Dense(neurons,activation='relu',input_shape=(x_train.shape[1],)))    

    for _ in range(layers-1): 
        model.add(Dense(neurons,activation='relu'))   

    model.add(Dense(1,activation='sigmoid')) 
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy']) 
    return model

In [ ]:
#  create a keras classifier 
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [ ]:
#  Defining GRID Search Parameter   
param_grid={
    'neurons':[16,32,64,128],
    'layers':[1,2,3],   #  hidden layer
    'epochs': [50,100]   
    }

In [ ]:
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3)
grid_result=grid.fit(x_train,y_train)  
print("Best : % f using %s " % (grid_result.best_score_,grid_result.best_params_)) 


f:\New folder\practice\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)




Epoch 1/50


250/250 [==============================] - 4s 5ms/step - loss: 0.5661 - accuracy: 0.7182
Epoch 2/50
250/250 [==============================] - 1s 4ms/step - loss: 0.4240 - accuracy: 0.8174
Epoch 3/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3914 - accuracy: 0.8346
Epoch 4/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3708 - accuracy: 0.8434
Epoch 5/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3597 - accuracy: 0.8485
Epoch 6/50
250/250 [==============================] - 1s 5ms/step - loss: 0.3533 - accuracy: 0.8501
Epoch 7/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3495 - accuracy: 0.8504
Epoch 8/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3472 - accuracy: 0.8535
Epoch 9/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3451 - accuracy: 0.8545
Epoch 10/50
250/250 [==============================] - 1s 4ms/step - loss: 0.3431 - accuracy: 0.

In [ ]:
print("Best : % f using %s " % (grid_result.best_score,grid_result.best_params_))